# 11. In-Place Array Tricks, Cyclic Sort and "Index as a Hash"

When values are in the range `1..n` (or `0..n`) and the array has size `n`, the **array itself can act as a hash map**: value `v` belongs at index `v - 1`. This gives O(n) time and **O(1) extra space**.

**Learning order:** ROTATE → CYCLIC SORT → MISSING NUMBERS → DUPLICATES → FIRST MISSING POSITIVE → SET MISMATCH → INTERVIEW THINKING

## 1. Rotate array by `k` (reverse trick)

Rotating right by `k`: reverse the whole array, then reverse the first `k` elements and the remaining `n - k` elements. O(n) time, O(1) space. Use `k %= n`.

`[1,2,3,4,5,6,7]`, `k = 3` → reverse all `[7,6,5,4,3,2,1]` → reverse first 3 `[5,6,7,...]` → reverse rest `[...,1,2,3,4]`.

In [1]:
def rotate(nums, k):
    n = len(nums)
    k %= n
    def reverse(l, r):
        while l < r:
            nums[l], nums[r] = nums[r], nums[l]
            l += 1
            r -= 1
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)
    return nums

print(rotate([1, 2, 3, 4, 5, 6, 7], 3))   # [5, 6, 7, 1, 2, 3, 4]

[5, 6, 7, 1, 2, 3, 4]


## 2. Cyclic sort (numbers `1..n`)

Put each number where it belongs: while `nums[i]` is not at its correct index (`nums[i] - 1`) **and** the target slot does not already hold the same value, **swap** it there. Each swap places one number correctly, so total swaps ≤ n → **O(n)**.

In [2]:
def cyclic_sort(nums):
    i = 0
    while i < len(nums):
        correct = nums[i] - 1
        if nums[i] != nums[correct]:
            nums[i], nums[correct] = nums[correct], nums[i]
        else:
            i += 1
    return nums

print(cyclic_sort([3, 1, 2]), cyclic_sort([5, 4, 3, 1, 2]))   # [1, 2, 3] [1, 2, 3, 4, 5]

[1, 2, 3] [1, 2, 3, 4, 5]


## 3. Find All Numbers Disappeared / Duplicates (sign marking)

Use the **sign** of `nums[v - 1]` as a "seen" flag.
- **Disappeared numbers:** for each value `v`, make `nums[|v| - 1]` negative. Indexes still positive at the end were never seen → `i + 1` is missing.
- **Duplicates:** if `nums[|v| - 1]` is already negative when we see `v`, then `v` is a duplicate.

In [3]:
def find_disappeared(nums):
    nums = nums[:]
    for v in nums:
        idx = abs(v) - 1
        if nums[idx] > 0:
            nums[idx] = -nums[idx]
    return [i + 1 for i, v in enumerate(nums) if v > 0]

def find_duplicates(nums):
    nums, res = nums[:], []
    for v in nums:
        idx = abs(v) - 1
        if nums[idx] < 0:
            res.append(abs(v))
        else:
            nums[idx] = -nums[idx]
    return sorted(res)

print(find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]), find_duplicates([4, 3, 2, 7, 8, 2, 3, 1]))   # [5, 6] [2, 3]

[5, 6] [2, 3]


## 4. First Missing Positive (hard)

The answer is in `1..n+1`. Cyclic sort the numbers in range `1..n` into place (ignore values `≤ 0` or `> n`). Then the first index `i` with `nums[i] != i + 1` gives the answer `i + 1`. O(n) time, O(1) space.

In [4]:
def first_missing_positive(nums):
    n = len(nums)
    for i in range(n):
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1

print(first_missing_positive([1, 2, 0]), first_missing_positive([3, 4, -1, 1]), first_missing_positive([7, 8, 9, 11, 12]))   # 3 2 1

3 2 1


## 5. Set Mismatch

One number appears twice, one is missing from `1..n`. Return `[duplicate, missing]`. Sign marking again: the value hitting an already negative slot is the duplicate; the index that stays positive is the missing number.

In [5]:
def find_error_nums(nums):
    nums, dup = nums[:], -1
    for v in nums:
        idx = abs(v) - 1
        if nums[idx] < 0:
            dup = abs(v)
        else:
            nums[idx] = -nums[idx]
    missing = next(i + 1 for i, v in enumerate(nums) if v > 0)
    return [dup, missing]

print(find_error_nums([1, 2, 2, 4]))   # [2, 3]

[2, 3]


## 6. Interview thinking

**Signals:** "values in `1..n`", "array of size `n`", "O(1) extra space", "find missing/duplicate", "without sorting".

**Say:** "Since values are in `1..n`, value `v` can live at index `v - 1`. I'll use the array as a hash map by swapping into place, or by marking visited indexes with a sign. O(n) time, O(1) space."

**Common mistakes:** infinite swap loop (forgot the `nums[i] != nums[correct]` check for duplicates), forgetting `abs()` when reading marked values, destroying the input without saying so (restore signs, or copy if allowed), assuming the range when it is not guaranteed.

## Quick Revision

- Rotate: reverse all, reverse first k, reverse rest.
- Cyclic sort: swap into `nums[i] - 1` until correct or duplicate.
- Sign marking: negate `nums[|v|-1]` as "seen".
- First missing positive: cyclic sort for `1..n`, scan for the first mismatch.
- Needs values in a known range and mutation of the input.